<a href="https://colab.research.google.com/github/huutai-cmyk/Homework-1/blob/main/btb4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import zipfile
from keras.models import Sequential
from keras.layers import Dense, Flatten, Conv2D, MaxPooling2D, Dropout, BatchNormalization

# ==========================================
# ĐIỂM SỬA LỖI: Gọi ImageDataGenerator từ tensorflow.keras
# ==========================================
from tensorflow.keras.preprocessing.image import ImageDataGenerator

thu_muc_goc = '/content/hình học'
thu_muc_anh_that = '/content/hinh_hoc_anh_that_v3'

os.makedirs(thu_muc_anh_that, exist_ok=True)

print("Đang xử lý dữ liệu ảnh 'hình học'")
for i in range(6):
    file_zip = os.path.join(thu_muc_goc, f'{i}.zip')
    thu_muc_giai_nen = os.path.join(thu_muc_anh_that, str(i))

    if os.path.exists(file_zip):
        with zipfile.ZipFile(file_zip, 'r') as zip_ref:
            zip_ref.extractall(thu_muc_giai_nen)
        print(f" Đã bung thành công file {i}.zip")
    else:
        print(f" Không tìm thấy file {i}.zip")

x_train_list = []
y_train_list = []

for root, dirs, files in os.walk(thu_muc_anh_that):
    for file_name in files:
        if file_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_path = os.path.join(root, file_name)

            img = cv2.imread(img_path)
            if img is not None:
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
                img_gray = cv2.resize(img_gray, (28, 28))
                img_gray = 255 - img_gray

                x_train_list.append(img_gray)

                try:
                    label = int(os.path.basename(root))
                except ValueError:
                    label = 0
                y_train_list.append(label)

print(f"Tổng số ảnh đã đọc thành công: {len(x_train_list)} ảnh.")

if len(x_train_list) == 0:
    print("LỖI:  không có ảnh.")
else:
    x_train = np.array(x_train_list)
    y_train = np.array(y_train_list)

    x_train_ready = x_train.reshape((x_train.shape[0], 28, 28, 1))
    x_train_ready = x_train_ready.astype('float32') / 255

    # XÂY DỰNG MẠNG CNN
    model = Sequential([
        Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(28, 28, 1)),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),

        Conv2D(64, kernel_size=(3, 3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),

        Flatten(),
        Dropout(0.2),

        Dense(128, activation='relu'),
        Dropout(0.2),

        Dense(6, activation='softmax')
    ])

    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    # TĂNG CƯỜNG DỮ LIỆU ĐỂ GIẢM NHẬN DIỆN SAI
    datagen = ImageDataGenerator(
        rotation_range=20,
        width_shift_range=0.1,
        height_shift_range=0.1,
        zoom_range=0.1,
        horizontal_flip=False
    )

    print("Đang huấn luyện Model AI ")
    model.fit(datagen.flow(x_train_ready, y_train, batch_size=4),
              steps_per_epoch=len(x_train_ready) // 4, epochs=15, verbose=1)

    print("Huấn luyện xong!")

    # LƯU MODEL ĐỂ WEB APP SỬ DỤNG
    model.save('/content/mo_hinh_hinh_hoc_cnn.h5')
    print("✅ Đã lưu Model thành công!")

--- Đang xử lý dữ liệu ảnh 'hình học' ---
✅ Đã bung thành công file 0.zip
✅ Đã bung thành công file 1.zip
✅ Đã bung thành công file 2.zip
✅ Đã bung thành công file 3.zip
✅ Đã bung thành công file 4.zip
✅ Đã bung thành công file 5.zip
🎯 Tổng số ảnh đã đọc thành công: 864 ảnh.
--- Đang huấn luyện Model AI (CNN nâng cấp)... ---
Epoch 1/15


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


216/216 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.9873 - loss: 0.0494
Epoch 2/15
216/216 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 1.0000 - loss: 2.5387e-05
Epoch 3/15
216/216 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 1.0000 - loss: 2.4745e-05
Epoch 4/15
216/216 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 1.0000 - loss: 1.4680e-05
Epoch 5/15
216/216 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 1.0000 - loss: 2.2025e-05
Epoch 6/15
216/216 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 1.0000 - loss: 1.7675e-06
Epoch 7/15
216/216 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - accuracy: 1.0000 - loss: 5.5831e-06
Epoch 8/15
216/216 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 1.0000 - loss: 3.0489e-06
Epoch 9/15
216/216 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 1.0000 - loss: 6.5814e-06
Epoch 10/15
216/216 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 1.0000 - loss: 3.3858e-06
Epoch 11/15
216/216 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 1.0000 - loss: 2.2532e-06
Epoch 12/15


--- Huấn luyện xong! ---
✅ Đã lưu Model thành công!


In [10]:

model.save('/content/mo_hinh_hinh_hoc.h5')
print("Đã lưu Model thành công!")

Đã lưu Model thành công!


In [11]:
%%writefile app.py
import streamlit as st
import cv2
import numpy as np
from keras.models import load_model
from PIL import Image

# 1. Tải mô hình AI đã lưu (đảm bảo file mo_hinh_hinh_hoc_cnn.h5 tồn tại)
@st.cache_resource # Gi giúp Streamlit không phải load lại model mỗi khi upload ảnh mới
def load_ai_model():
    # Sửa tên model cho khớp với tên model mới chúng ta vừa lưu
    return load_model('/content/mo_hinh_hinh_hoc_cnn.h5')

model = load_ai_model()

# Khai báo các nhãn đúng thứ tự (6 hình)
# 0: Binh hanh, 1: Vuong, 2: Tron, 3: Thang, 4: Chu nhat, 5: Tam giac
shapes = ['Hinh binh hanh', 'Hinh vuong', 'Hinh tron', 'Hinh thang', 'Hinh chu nhat', 'Hinh tam giac']

# 2. Xây dựng giao diện Web
st.title("📐 Ứng Dụng Nhận Diện Hình Học Cơ Bản (Bản Nâng Cấp)")
st.write("Hãy tải lên một bức ảnh hình học (nền đen nét trắng hoặc ngược lại), AI sẽ đoán đó là hình gì nhé!")

# Nút upload ảnh
uploaded_file = st.file_uploader("Chọn một bức ảnh...", type=["jpg", "jpeg", "png"])

if uploaded_file is not None:
    # --- HIỂN THỊ ẢNH GỐC LÊN WEB ---
    # Đọc file ảnh dưới dạng byte rồi chuyển thành mảng Numpy
    file_bytes = np.asarray(bytearray(uploaded_file.read()), dtype=np.uint8)

    # Giải mã thành ảnh BGR giống hệt như hàm cv2.imread() của em
    img = cv2.imdecode(file_bytes, 1)

    st.image(img, channels="BGR", caption="Ảnh bạn vừa tải lên", use_container_width=True)

    # --- KHỐI XỬ LÝ ẢNH CHUẨN CỦA EM ---
    st.write("⚙️ Đang phân tích hình ảnh...")

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    img_gray = cv2.resize(img_gray, (28, 28))
    img_gray = 255 - img_gray # Đảo màu

    # CẬP NHẬT Ở ĐÂY: Sửa lại reshape để phù hợp với mạng CNN
    # Định dạng mới: (số ảnh, 28, 28, 1) thay vì (số ảnh, 784)
    img_ready = img_gray.reshape((1, 28, 28, 1))
    img_ready = img_ready.astype('float32') / 255 # Chuẩn hóa ma trận điểm ảnh về 0-1

    # --- AI DỰ ĐOÁN ---
    preds = model.predict(img_ready)
    digit = np.argmax(preds)
    do_chinh_xac = np.max(preds) * 100

    # --- IN KẾT QUẢ RA MÀN HÌNH ---
    st.success(f"🎯 Kết quả dự đoán: **{shapes[digit]}**")
    st.info(f"📊 Độ tự tin của AI: {do_chinh_xac:.2f}%")

Overwriting app.py


In [12]:
# 1. Cài đặt Streamlit
!pip install -q streamlit

# 2. Tải công cụ Cloudflare Tunnel (Vượt qua màn hình IP)
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

# 3. Chạy Streamlit ngầm
!nohup streamlit run app.py > streamlit.log 2>&1 &

# 4. Chạy Cloudflare Tunnel và lấy link trực tiếp
import time
print("⏳ Đang khởi tạo máy chủ Web, em đợi khoảng 5-7 giây nhé...")
!nohup ./cloudflared tunnel --url http://localhost:8501 > cloudflared.log 2>&1 &
time.sleep(7) # Đợi hệ thống sinh ra link

print("✅ Xong rồi! Em bấm vào đường link đuôi '.trycloudflare.com' bên dưới để vào thẳng Web luôn nhé:")
!grep -o 'https://.*\.trycloudflare.com' cloudflared.log

cloudflared: Text file busy
⏳ Đang khởi tạo máy chủ Web, em đợi khoảng 5-7 giây nhé...
✅ Xong rồi! Em bấm vào đường link đuôi '.trycloudflare.com' bên dưới để vào thẳng Web luôn nhé:
https://mrna-discrimination-wise-initiatives.trycloudflare.com
